In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from preprocessing import dt_profiles_rating_df, preprocess_text

C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\jeffr\AppData\Roaming\Python\Python311\site-packages\urllib3\connectionpool.py:1095: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ud-anthony.vpnstores.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [2]:
### USER BASED FILLTERING ###
text_colums_user = ['gender','skin_type_face', 'hair_issue', 
                'skin_type_body', 'allergy_history', 'preferred_products', 
                'avoided_products', 'specific_needs']
                
for column in text_colums_user:
    dt_profiles_rating_df[column] = dt_profiles_rating_df[column].apply(preprocess_text)

In [3]:
# convert skin type, hair issue, skin type body to numeric value (int)
def convert_skin_type_face(skin_type): 
    skin_type_dict = {'normal': 0, 'kering': 1, 'minyak': 2, 'sensitif': 3, 'kombinasi': 4}
    return skin_type_dict.get(skin_type, 0)

def convert_hair_issue(hair_issue): 
    hair_issue_dict = {'normal': 0, 'ketombe': 1, 'kering': 2, 'minyak': 3, 'rontok': 4, 'cabang': 5}
    return hair_issue_dict.get(hair_issue, 0)

def convert_skin_type_body(skin_type): 
    skin_type_dict = {'normal': 0, 'kering': 1, 'minyak': 2, 'kombinasi': 3} 
    return skin_type_dict.get(skin_type, 0)

dt_profiles_rating_df["skin_type_face"] = dt_profiles_rating_df["skin_type_face"].apply(convert_skin_type_face) 
dt_profiles_rating_df["hair_issue"] = dt_profiles_rating_df["hair_issue"].apply(convert_hair_issue) 
dt_profiles_rating_df["skin_type_body"] = dt_profiles_rating_df["skin_type_body"].apply(convert_skin_type_body)

In [4]:
# Precompute user vectors
user_vectors = dt_profiles_rating_df.groupby('user_id')[['skin_type_face', 'hair_issue', 'skin_type_body']].mean().round(2)
user_vectors.reset_index(inplace=True)
display(user_vectors)

user_vectors = user_vectors[user_vectors['user_id'].isin(dt_profiles_rating_df['user_id'].unique())]
user_ids = user_vectors['user_id']
user_vectors = user_vectors.drop('user_id', axis=1)
user_similarities = cosine_similarity(user_vectors)
user_similarities = pd.DataFrame(user_similarities, index=user_ids, columns=user_ids).round(2)
display(user_similarities)

# Get unique items
items = dt_profiles_rating_df['product_id'].unique()
display(items)

,user_id,skin_type_face,hair_issue,skin_type_body
0,1,0.0,0.0,0.0
1,7,0.0,3.0,0.0
2,8,3.0,2.0,1.0
3,11,0.0,1.0,2.0
4,12,0.0,0.0,0.0
...,...,...,...,...
294,360,0.0,1.0,0.0
295,361,0.0,1.0,0.0
296,362,0.0,1.0,0.0
297,363,2.0,2.0,0.0


user_id,1,7,8,11,12,13,14,15,16,17,...,355,356,357,358,359,360,361,362,363,364
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,0.0,1.00,0.53,0.45,0.0,0.45,0.45,0.71,1.00,1.00,...,1.00,0.24,0.78,1.00,1.00,1.00,1.00,1.00,0.71,1.00
8,0.0,0.53,1.00,0.48,0.0,0.96,0.96,0.94,0.53,0.53,...,0.53,0.91,0.92,0.53,0.53,0.53,0.53,0.53,0.94,0.53
11,0.0,0.45,0.48,1.00,0.0,0.20,0.20,0.32,0.45,0.45,...,0.45,0.11,0.35,0.45,0.45,0.45,0.45,0.45,0.32,0.45
12,0.0,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,0.0,1.00,0.53,0.45,0.0,0.45,0.45,0.71,1.00,1.00,...,1.00,0.24,0.78,1.00,1.00,1.00,1.00,1.00,0.71,1.00
361,0.0,1.00,0.53,0.45,0.0,0.45,0.45,0.71,1.00,1.00,...,1.00,0.24,0.78,1.00,1.00,1.00,1.00,1.00,0.71,1.00
362,0.0,1.00,0.53,0.45,0.0,0.45,0.45,0.71,1.00,1.00,...,1.00,0.24,0.78,1.00,1.00,1.00,1.00,1.00,0.71,1.00


array([  0, 215,  22, 311, 286,  82, 140, 112,  63,  51,  21,  18,   5,
         8,  68,  92,  85, 216, 307, 280, 276, 236,  58,  24, 300, 315,
         7, 294, 134,  42,  43, 123, 167, 168, 199,  84, 283,  16, 309,
        47,  20, 263,  35, 135, 219,  61,  78,  93,  62,  39, 282, 316,
       165, 169, 261, 191,   6, 295, 122, 314,  46, 312, 306,  59, 188,
        36, 138, 298,  81, 166, 310,  56,  23, 161, 207, 252,  40, 179,
        95, 141, 163, 170,  14, 177,  87, 120,  64,  77,  44,  26,  60,
       208,  94, 302, 228, 193, 147,  32,   4, 171,  69,  79, 114, 274,
       260, 160,   9, 304, 234, 218, 285, 299,  15,  37,  25,  83, 121,
       156, 159,  66, 164,  57, 211,  13, 305,  50, 116, 149,  33,  12,
       175, 130, 273, 217,  30, 313, 259, 137,  73, 232,  45,  65,  75,
        34, 187,  67, 265, 255,  28, 233, 214,  74,  96, 278, 267, 258,
       195, 229, 292, 254, 251, 227, 178, 139,  99,  98,  97,  88,  71,
        55, 146, 262, 212, 279,  49, 246, 173, 210, 124, 287,  4

In [5]:
# Function to get recommendations
def get_user_based_recommendations(user_id):
    predictions = {}
    similarity_sum = user_similarities.loc[user_id].sum()
    
    if similarity_sum > 0:
        for item in items:
            other_user_ratings = dt_profiles_rating_df[dt_profiles_rating_df['product_id'] == item]
            rating_sum = 0
            weight_sum = 0
            for other_user_id in other_user_ratings['user_id']:
                if other_user_id != user_id:
                    rating = other_user_ratings[other_user_ratings['user_id'] == other_user_id]['rating'].values[0]
                    similarity = user_similarities.loc[user_id, other_user_id]
                    rating_sum += rating * similarity
                    weight_sum += similarity
            if weight_sum > 0:
                predictions[item] = rating_sum / weight_sum
            
    recommendations = sorted(predictions, key=predictions.get, reverse=True)[:10]
    return recommendations

# Display recommendations for each user
unique_user_ids = dt_profiles_rating_df['user_id'].unique()

In [6]:
for user_id in unique_user_ids:
    recommendations = get_user_based_recommendations(user_id)
    user_name = dt_profiles_rating_df[dt_profiles_rating_df['user_id'] == user_id]['user_name'].values[0]  # Assuming 'user_name' column exists
    print(f"Top 10 recommended products for user_id = {user_id} ({user_name})")
    print(f"Product IDs: {recommendations}")
    print()

Top 10 recommended products for user_id = 1 (Operator)
Product IDs: []

Top 10 recommended products for user_id = 7 (Dini Sipahutar)
Product IDs: [137, 151, 154, 276, 78, 95, 77, 13, 305, 50]

Top 10 recommended products for user_id = 8 (Gladys)
Product IDs: [52, 95, 260, 83, 232, 45, 246, 287, 86, 148]

Top 10 recommended products for user_id = 11 (Suandika Napitupulu)
Product IDs: [83, 275, 142, 78, 177, 77, 13, 50, 137, 232]

Top 10 recommended products for user_id = 12 (Elisa)
Product IDs: []

Top 10 recommended products for user_id = 13 (Emy Sonia Sinambela)
Product IDs: [49, 142, 145, 152, 126, 78, 177, 77, 260, 13]

Top 10 recommended products for user_id = 14 (Stefhani Kezia)
Product IDs: [49, 142, 145, 152, 126, 78, 177, 77, 260, 13]

Top 10 recommended products for user_id = 15 (Josep Phyto Napitupulu)
Product IDs: [78, 177, 13, 187, 279, 287, 52, 190, 277, 148]

Top 10 recommended products for user_id = 16 (Samuel Simanjuntak)
Product IDs: [137, 151, 154, 78, 95, 77, 13, 305